# 加法进位实验


<img src="https://github.com/JerrikEph/jerrikeph.github.io/raw/master/Learn2Carry.png" width=650>

In [2]:
!pip install tqdm

In [3]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm


## 数据生成
我们随机在 `start->end`之间采样除整数对`(num1, num2)`，计算结果`num1+num2`作为监督信号。

* 首先将数字转换成数字位列表 `convertNum2Digits`
* 将数字位列表反向
* 将数字位列表填充到同样的长度 `pad2len`


In [4]:
def gen_data_batch(batch_size, start, end):
    '''在(start, end)区间采样生成一个batch的整型的数据
    Args :
        batch_size: batch_size
        start: 开始数值
        end: 结束数值
    '''
    numbers_1 = np.random.randint(start, end, batch_size)
    numbers_2 = np.random.randint(start, end, batch_size)
    results = numbers_1 + numbers_2
    return numbers_1, numbers_2, results

def convertNum2Digits(Num):
    '''将一个整数转换成一个数字位的列表,例如 133412 ==> [1, 3, 3, 4, 1, 2]
    '''
    strNum = str(Num)
    chNums = list(strNum)
    digitNums = [int(o) for o in strNum]
    return digitNums

def convertDigits2Num(Digits):
    '''
    将数字列表直接拼接成整数
    [1, 3, 3, 4, 1, 2] → "133412" → 133412
    '''
    digitStrs = [str(o) for o in Digits]
    numStr = ''.join(digitStrs)
    Num = int(numStr)
    return Num

def pad2len(lst, length, pad=0):
    '''将一个列表用`pad`填充到`length`的长度 例如 pad2len([1, 3, 2, 3], 6, pad=0) ==> [1, 3, 2, 3, 0, 0]
    '''
    lst+=[pad]*(length - len(lst))
    return lst

def results_converter(res_lst):
    '''
    对每个数字列表先做 reversed() 反转
    再调用 convertDigits2Num 转换为整数
    res_lst = [[1,3,3,4,1,2], [5,6,7]]
    reversed → [[2,1,4,3,3,1], [7,6,5]]
    转换整数 → [214331, 765]
    
    Args:
        res_lst: shape(b_sz, len(digits))
    '''
    res = [reversed(digits) for digits in res_lst]
    return [convertDigits2Num(digits) for digits in res]

def prepare_batch(Nums1, Nums2, results, maxlen):
    '''准备一个batch的数据，将数值转换成反转的数位列表并且填充到固定长度
    Args:
        Nums1: shape(batch_size,)
        Nums2: shape(batch_size,)
        results: shape(batch_size,)
        maxlen:  type(int)
    Returns:
        Nums1: shape(batch_size, maxlen)
        Nums2: shape(batch_size, maxlen)
        results: shape(batch_size, maxlen)

    maxlen = 5
    Nums1 = [12, 3]      # 两个样本
    Nums2 = [34, 5]
    results = [46, 8]    # 12+34=46, 3+5=8
    '''
    Nums1 = [convertNum2Digits(o) for o in Nums1]  # Nums1: 12 → [1,2]      3 → [3]
    Nums2 = [convertNum2Digits(o) for o in Nums2]  
    results = [convertNum2Digits(o) for o in results] # results: 46 → [4,6]    8 → [8]
    
    Nums1 = [list(reversed(o)) for o in Nums1]  # Nums1: [1,2] → [2,1]    [3] → [3]
    Nums2 = [list(reversed(o)) for o in Nums2]
    results = [list(reversed(o)) for o in results]
    
    Nums1 = [pad2len(o, maxlen) for o in Nums1]  # Nums1: [2,1] → [2,1,0,0,0]    [3] → [3,0,0,0,0]
    Nums2 = [pad2len(o, maxlen) for o in Nums2]
    results = [pad2len(o, maxlen) for o in results]
    
    return Nums1, Nums2, results

# 建模过程， 按照图示完成建模

In [7]:
'''
SimpleRNNCell单个时间步的计算单元:
h_t = tanh(W_h * h_{t-1} + W_x * x_t + b)
'''

class myRNNModel(keras.Model):
    def __init__(self):
        super(myRNNModel, self).__init__()
        self.embed_layer = tf.keras.layers.Embedding(10, 32, 
                                                    batch_input_shape=[None, None])
        
        self.rnncell = tf.keras.layers.SimpleRNNCell(64)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)  # 将 RNNCell 按时间步循环执行
        # return_sequences=True 返回所有时间步的输出 (batch, seq_len, 64)
        # False：只返回最后时间步的输出 (batch, 64)
        
        self.dense = tf.keras.layers.Dense(10)  # 10分类
        # 全连接层，将64维特征映射到10维
        
    @tf.function
    def call(self, num1, num2):
        '''
        此处完成上述图中模型        
        '''
        embed1=self.embed_layer(num1)  # (batch, seq_len, 32)
        embed2=self.embed_layer(num2)

        combined=tf.concat([embed1,embed2],axis=-1)  # (batch, seq_len, 64)

        rnn_out=self.rnn_layer(combined)  # (batch, seq_len, 64)

        logits=self.dense(rnn_out) # (batch, seq_len, 10)
        return logits

In [8]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    return tf.reduce_mean(losses)

'''
输入 x, y, label
    ↓
[GradientTape 开始录像]
    ↓
前向传播: model(x, y) → logits
    ↓
计算损失: compute_loss(logits, label) → loss
    ↓
[GradientTape 停止录像]
    ↓
计算梯度: tape.gradient(loss, variables) → grads
    ↓
更新参数: optimizer.apply_gradients(zip(grads, variables))
    ↓
返回 loss
'''

@tf.function # 静态计算图
def train_one_step(model, optimizer, x, y, label):
    # 创建一个记录器，追踪所有在块内执行的操作，用于后续自动求导
    with tf.GradientTape() as tape:
        logits = model(x, y)  # 将输入传递给模型，得到预测输出
        # logits shape: (batch_size, maxlen, 10)
        # 每个时间步的10维向量表示该位置预测0-9的未归一化分数
        loss = compute_loss(logits, label)  # 计算预测值与真实标签之间的交叉熵损失

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)  # 用记录器计算损失对模型参数的梯度
    # grads[i] = ∂loss / ∂(model.trainable_variables[i])
    
    optimizer.apply_gradients(zip(grads, model.trainable_variables)) # 用梯度更新模型参数
    # zip(grads, model.trainable_variables)：将梯度和参数配对
    return loss

def train(steps, model, optimizer):
    loss = 0.0
    accuracy = 0.0
    for step in range(steps):
        datas = gen_data_batch(batch_size=200, start=0, end=555555555)
        # datas 包含 (Nums1, Nums2, results) 三个数组，每个长度200

        # 2. 预处理：转换为反转+填充的数字序列
        Nums1, Nums2, results = prepare_batch(*datas, maxlen=11)

        # 训练一步
        loss = train_one_step(model, optimizer, 
                              tf.constant(Nums1, dtype=tf.int32), 
                              tf.constant(Nums2, dtype=tf.int32),
                              tf.constant(results, dtype=tf.int32))
        if step%50 == 0:
            print('step', step, ': loss', loss.numpy())

    return loss

def evaluate(model):
    datas = gen_data_batch(batch_size=2000, start=555555555, end=999999999)
    Nums1, Nums2, results = prepare_batch(*datas, maxlen=11)
    logits = model(tf.constant(Nums1, dtype=tf.int32), tf.constant(Nums2, dtype=tf.int32))
    logits = logits.numpy()
    pred = np.argmax(logits, axis=-1)
    res = results_converter(pred)
    for o in list(zip(datas[2], res))[:20]:
        print(o[0], o[1], o[0]==o[1])

    print('accuracy is: %g' % np.mean([o[0]==o[1] for o in zip(datas[2], res)]))


In [9]:
optimizer = optimizers.Adam(0.001)
model = myRNNModel()

In [10]:
train(3000, model, optimizer)
evaluate(model)

step 0 : loss 2.2981787
step 50 : loss 1.9349965
step 100 : loss 1.9220157
step 150 : loss 1.8954653
step 200 : loss 1.9016482
step 250 : loss 1.8984257
step 300 : loss 1.8840507
step 350 : loss 1.8844342
step 400 : loss 1.8873814
step 450 : loss 1.8944751
step 500 : loss 1.8876129
step 550 : loss 1.8751194
step 600 : loss 1.8934846
step 650 : loss 1.8925706
step 700 : loss 1.8821063
step 750 : loss 1.8813214
step 800 : loss 1.8751718
step 850 : loss 1.8666424
step 900 : loss 1.8716828
step 950 : loss 1.8749454
step 1000 : loss 1.86216
step 1050 : loss 1.8770379
step 1100 : loss 1.8734602
step 1150 : loss 1.8691202
step 1200 : loss 1.8692813
step 1250 : loss 1.8718643
step 1300 : loss 1.8619382
step 1350 : loss 1.8615981
step 1400 : loss 1.8489007
step 1450 : loss 1.8156275
step 1500 : loss 1.7621031
step 1550 : loss 1.6982774
step 1600 : loss 1.6699831
step 1650 : loss 1.6339897
step 1700 : loss 1.5455365
step 1750 : loss 1.3148009
step 1800 : loss 1.1108986
step 1850 : loss 0.961259
